# 1.6 포인트 적립

영수증 사진을 읽고, 주문 내역과 비교한 뒤 고객이 확인한 경우에만 포인트를 적립합니다.

- 입력: 영수증 사진과 고객 정보가 담긴 `SessionState`
- 처리: 이미지 읽기 → 거래·중복 확인 → 고객 확인 → 테스트 적립
- 출력: 적립 결과가 담긴 `SessionState`

예: 모찌카페 영수증의 주문번호와 결제금액을 읽고 맞는지 확인합니다.

프로젝트 루트에서 실행합니다. Gemini 키가 환경변수에 필요하며 영수증 이미지가 외부 API로 전송됩니다. 저장된 실행 결과는 아직 없습니다. 실제 고객 파일은 수정하지 않습니다.

```mermaid
flowchart LR
    input["Input<br/>SessionState"] --> ocr["영수증 읽기<br/>generate_receipt_data"]
    ocr --> valid["거래와 중복 검증<br/>validate_receipt"]
    valid --> confirm["고객 확인 요청<br/>request_receipt_confirmation"]
    confirm --> add["확인 후 적립<br/>add_receipt_points"]
    add --> result["적립 결과 안내<br/>format_point_result"]
    result --> output["Output<br/>SessionState"]
    classDef io fill:#eeeeee,stroke:#999999,color:#333333;
    classDef python fill:#dbeafe,stroke:#3b82f6,color:#1e3a8a;
    class input,output io;
    class valid,confirm,add,result python;
```

## 1. 스키마 설정

고객과 세션은 공통 스키마를 사용합니다. 영수증 정보는 이번 세션의 `receipt`에 보관합니다.

In [1]:
# 1. 경로 준비: 프로젝트 루트에서 공통 스키마를 가져옵니다.
import sys
import os
sys.path.append(os.getcwd())

# 2. 스키마 확인: 고객 입력과 대화 상태의 공통 형식을 사용합니다.
from schemas.session_state import CustomerInput, SessionState
print("고객 입력:", list(CustomerInput.model_fields))
print("세션 상태:", list(SessionState.model_fields))
from schemas.session_state import CustomerProfile, ReceiptState
print("고객 정보:", list(CustomerProfile.model_fields))

고객 입력: ['text', 'has_image']
세션 상태: ['session_id', 'customer_id', 'customer', 'input', 'intent', 'previous_intent', 'history', 'search', 'recommendation', 'coupon', 'order', 'receipt', 'answer']
고객 정보: ['customer_id', 'name', 'birth_year', 'membership_level', 'allergies', 'points', 'stamps', 'preferences']


## 2. 모델 준비

`0_2`와 같은 Gemini 연결로 사진을 읽습니다.

In [2]:
# 1. 도구 준비: 이미지와 주문 데이터를 읽고 모델에 전달할 도구입니다.
import base64
import json
from pathlib import Path
from openai import OpenAI

# 2. 모델 연결: 환경변수의 키로 Gemini를 준비합니다.
ocr_model = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
ocr_client = OpenAI(api_key=os.environ["GEMINI_API_KEY"], base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
print("영수증 읽기 모델:", ocr_model)

영수증 읽기 모델: gemini-2.5-flash


## 3. 관련 프롬프트 불러오기

읽은 값을 추측하지 않도록 영수증 읽기 지침을 확인합니다.

In [3]:
# 1. 지침 읽기: 영수증을 JSON으로 읽는 지침을 불러옵니다.
receipt_prompt = Path("prompts/receipt_to_json.md").read_text(encoding="utf-8")
print(receipt_prompt)

# 영수증 내용 읽기

이미지에 보이는 영수증 내용을 읽어 JSON 객체 하나로 반환합니다. 보이지 않거나 확실하지 않은 값은 추측하지 말고 null로 둡니다.

- `store_name`: 매장 이름 문자열 또는 null
- `receipt_number`: 거래번호 문자열 또는 null
- `purchased_at`: 영수증에 적힌 일시 문자열 또는 null
- `items`: 메뉴별 객체 배열. 각 객체는 `name`(메뉴명 문자열 또는 null), `quantity`(수량 정수 또는 null), `unit_price`(단가 정수 또는 null), `amount`(금액 정수 또는 null)를 담습니다.
- `total`: 합계 정수 또는 null
- `payment_amount`: 결제금액 정수 또는 null
- `other_text`: 위 항목에 담지 않은 보이는 글자 문자열 배열

JSON 이외의 설명이나 코드 블록은 출력하지 않습니다.



## 4. 사용자 input 준비

영수증 사진과 가상 주문 목록을 사용합니다. 고객 파일을 변경하지 않으며, 중복 기록과 포인트도 메모리에서만 사용합니다.

In [4]:
# 1. 자료 준비: 샘플 영수증과 가상 주문을 확인합니다.
image_path = Path("data/sample_receipts/receipt-001.png")
orders = json.loads(Path("data/orders.json").read_text(encoding="utf-8"))
processed_receipts = set()
print("영수증 사진:", image_path)
print("주문 자료 개수:", len(orders))
print("이미 적립한 영수증 수:", len(processed_receipts))

영수증 사진: data/sample_receipts/receipt-001.png
주문 자료 개수: 40
이미 적립한 영수증 수: 0


## 5. 함수 만들기

실제 영수증 사진을 모델에 보내고, 확인 전과 확인 후의 상태를 구분합니다.

In [5]:
def generate_receipt_data(state: SessionState):
    """세션의 영수증 사진을 Gemini에 보내, 읽은 내용을 상태에 담아 돌려줍니다."""
    # 1. 이미지 준비: PNG 파일을 모델에 보낼 문자열로 바꿉니다.
    encoded = base64.b64encode(Path(state.receipt.image_path).read_bytes()).decode("utf-8")

    # 2. 모델 호출: 영수증 읽기 지침과 실제 이미지를 함께 보냅니다.
    response = ocr_client.chat.completions.create(model=ocr_model, response_format={"type": "json_object"}, messages=[{"role": "user", "content": [{"type": "text", "text": receipt_prompt}, {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{encoded}"}}]}])
    receipt = json.loads(response.choices[0].message.content)

    # 3. 결과 저장: 확실하지 않거나 형식이 맞지 않는 값은 미확인으로 보관합니다.
    state.receipt.store_name = receipt.get("store_name") if isinstance(receipt.get("store_name"), str) else None
    state.receipt.receipt_number = receipt.get("receipt_number") if isinstance(receipt.get("receipt_number"), str) else None
    state.receipt.purchased_at = receipt.get("purchased_at") if isinstance(receipt.get("purchased_at"), str) else None

    ## 결제금액은 음수가 아닌 정수일 때만 저장합니다.
    amount = receipt.get("payment_amount")
    state.receipt.payment_amount = amount if type(amount) is int and amount >= 0 else None

    ## 새로 읽은 영수증은 검증과 고객 확인을 다시 받도록 초기화합니다.
    state.receipt.verification = None
    state.receipt.customer_confirmed = None
    state.receipt.status = "pending"

    return state

In [6]:
def validate_receipt(state: SessionState):
    """읽은 영수증과 주문 자료를 비교해, 적립 가능 여부와 이유를 상태에 담아 돌려줍니다."""
    # 1. 입력 확인: 필수 항목과 중복 처리 이력, 고객 정보가 있는지 확인합니다.
    receipt = state.receipt
    if receipt.store_name != "모찌카페" or not receipt.receipt_number or receipt.payment_amount is None or receipt.payment_amount <= 0:
        receipt.verification = "failed"
        receipt.reason = "매장명, 주문번호 또는 결제금액을 확인해 주세요."
    elif state.customer is None or state.customer.points is None:
        receipt.verification = "failed"
        receipt.reason = "고객과 현재 포인트를 확인해 주세요."
    elif receipt.receipt_number in processed_receipts:
        receipt.verification = "failed"
        receipt.reason = "이미 적립한 영수증입니다."
    else:
        # 2. 거래 확인: 같은 고객의 주문번호와 결제금액이 맞는지 확인합니다.
        matches = [order for order in orders if order["order_id"] == receipt.receipt_number and order["customer_id"] == state.customer.customer_id and order["total_price"] == receipt.payment_amount]

        ## 고객 주문과 결제금액의 일치 여부를 결과와 이유에 담습니다.
        if matches:
            receipt.verification = "passed"
            receipt.reason = "주문과 결제금액이 일치합니다."
        else:
            receipt.verification = "failed"
            receipt.reason = "일치하는 고객 주문과 결제금액을 찾지 못했습니다."

    # 3. 상태 반환: 확인 결과를 고객 확인 단계에 전달합니다.

    return state

In [7]:
def request_receipt_confirmation(state: SessionState):
    """검증한 영수증을 받아, 고객이 확인할 내용을 답변에 담아 돌려줍니다."""
    # 1. 확인 요청: 검증 성공 시 읽은 주문번호와 금액을 보여줍니다.
    receipt = state.receipt
    if receipt.verification == "passed":
        state.answer = {"text": f"주문번호 {receipt.receipt_number}, 결제금액 {receipt.payment_amount:,}원이 맞나요?", "model": "확인 안내"}
    else:
        state.answer = {"text": receipt.reason or "영수증을 확인해 주세요.", "model": "확인 안내"}

    # 2. 상태 반환: 확인 전에는 포인트를 변경하지 않습니다.

    return state

In [8]:
def add_receipt_points(state: SessionState, points_to_add: int | None):
    """고객 확인과 지정된 테스트 포인트를 받아, 중복 확인 후 가상 잔액을 갱신합니다."""
    # 1. 처리 조건: 적립 기준이 미정이므로 연습할 포인트를 명시해야 합니다.
    receipt = state.receipt
    if receipt.status == "completed" or receipt.receipt_number in processed_receipts:
        receipt.status = "failed"
        receipt.reason = "이미 적립한 영수증입니다."
    elif receipt.verification != "passed" or receipt.customer_confirmed is not True:
        receipt.status = "failed"
        receipt.reason = "검증과 고객 확인이 필요합니다."
    elif type(points_to_add) is not int or points_to_add <= 0:
        receipt.status = "failed"
        receipt.reason = "연습에 사용할 적립 포인트를 지정해 주세요."
    else:
        # 2. 테스트 적립: 세션의 가상 잔액과 메모리 중복 기록만 함께 갱신합니다.
        state.customer.points += points_to_add
        processed_receipts.add(receipt.receipt_number)

        ## 중복 기록을 남긴 뒤 이번 테스트 적립 결과를 보관합니다.
        receipt.status = "completed"
        receipt.points_to_add = points_to_add
        receipt.reason = "테스트 포인트를 적립했습니다."

    # 3. 상태 반환: 실제 고객 파일에는 아무것도 쓰지 않습니다.

    return state

In [9]:
def format_point_result(state: SessionState):
    """포인트 처리 상태를 받아, 적립 결과나 실패 이유를 고객 답변에 담아 돌려줍니다."""
    # 1. 결과 안내: 실제 갱신한 경우에만 추가 포인트와 잔액을 보여줍니다.
    receipt = state.receipt
    if receipt.status == "completed":
        message = f"테스트 포인트 {receipt.points_to_add}점을 적립했습니다. 현재 잔액은 {state.customer.points}점입니다."
    else:
        message = receipt.reason or "아직 적립하지 않았습니다."

    ## 적립 성공 또는 실패 안내를 고객 답변에 담습니다.
    state.answer = {"text": message, "model": "결과 안내"}

    # 2. 상태 반환: 고객에게 보여 줄 결과를 전달합니다.

    return state

## 6. 연결 테스트

### 테스트 1 — 사진 읽기와 확인 요청

Gemini에 실제 영수증 사진을 전송합니다. 검증에 성공해도 이 턴에서는 적립하지 않습니다.

In [10]:
# 1. 입력 준비: 가상 고객과 실제 샘플 사진 경로를 상태에 담습니다.
state = SessionState(customer=CustomerProfile(customer_id="cust_001", points=0), input=CustomerInput(text="이 영수증으로 포인트를 적립해 주세요", has_image=True))
state.receipt = ReceiptState(image_path=str(image_path))
print("입력:", state.input.text, "/", state.receipt.image_path)

# 2. 함수 호출: 사진을 읽고, 주문과 비교해 확인 문장을 만듭니다.
print("\n함수: generate_receipt_data → validate_receipt → request_receipt_confirmation")
state = generate_receipt_data(state)
print("읽은 주문번호:", state.receipt.receipt_number)
print("읽은 결제금액:", state.receipt.payment_amount)

## 읽은 영수증을 주문 자료와 비교하고 고객 확인 질문을 만듭니다.
state = validate_receipt(state)
print("검증:", state.receipt.verification, state.receipt.reason)
state = request_receipt_confirmation(state)

# 3. 출력 확인: 확인 전 잔액과 고객에게 보낼 질문을 확인합니다.
print("\n답변:", state.answer["text"])
print("확인 전 잔액:", state.customer.points)

입력: 이 영수증으로 포인트를 적립해 주세요 / data/sample_receipts/receipt-001.png

함수: generate_receipt_data → validate_receipt → request_receipt_confirmation


읽은 주문번호: ord_20260905_0001
읽은 결제금액: 9200
검증: passed 주문과 결제금액이 일치합니다.

답변: 주문번호 ord_20260905_0001, 결제금액 9,200원이 맞나요?
확인 전 잔액: 0


### 테스트 2 — 고객 확인과 테스트 적립

첫 테스트에서 검증에 성공했을 때만 고객 확인을 가정합니다. **100점은 실습을 위해 직접 지정한 값이며 실제 적립률이 아닙니다.** 실제 포인트 파일은 바뀌지 않습니다.

In [11]:
# 1. 입력 준비: 같은 세션에서 고객의 확인 답변과 연습용 포인트를 직접 지정합니다.
state.input = CustomerInput(text="네, 맞아요")
state.receipt.customer_confirmed = state.input.text == "네, 맞아요"
practice_points = 100
print("입력:", state.input.text)
print("연습용 적립 포인트:", practice_points)

# 2. 함수 호출: 검증 결과와 고객 확인, 중복 기록을 확인해 테스트 적립합니다.
print("\n함수: add_receipt_points → format_point_result")
state = add_receipt_points(state, practice_points)
state = format_point_result(state)

# 3. 출력 확인: 실제 파일이 아닌 세션에 저장한 결과를 확인합니다.
print("\n결과:", state.answer["text"])
print("처리 상태:", state.receipt.status)

입력: 네, 맞아요
연습용 적립 포인트: 100

함수: add_receipt_points → format_point_result

결과: 테스트 포인트 100점을 적립했습니다. 현재 잔액은 100점입니다.
처리 상태: completed
